In [ ]:
"""MFCC_Arduino_Training_v3.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1LR4QsOpBERJfjjQxqW2M7W-Ei15f_XdR

# Keyword Spotting — Training Notebook
## From audio recordings to a deployable TFLite model for Arduino

We're classifying 4 sounds: **clap**, **tap**, **snap**, and **silence**.

The tricky part here is that the MFCC extraction on Arduino (CMSIS-DSP) has to produce
the exact same numbers as what we compute in this notebook. We learned this the hard way —
our first version used `librosa.feature.mfcc()` which applies Slaney normalization and
orthonormal DCT by default, and the Arduino values were completely off.

So in this version we build the whole MFCC pipeline from scratch using scipy, matching
the Arduino code step by step.

## 0 — Install dependencies
"""

In [ ]:
!pip install -q librosa soundfile tensorflow numpy matplotlib scikit-learn seaborn pydub
!apt-get install -q ffmpeg

In [ ]:
"""## 1 — Parameters

These have to match the `#define`s in the Arduino sketch exactly.
If you change anything here, you need to change the sketch too.
"""

In [ ]:
import numpy as np
import os

In [ ]:
# Audio settings
SAMPLE_RATE   = 16000
CLIP_DURATION = 1.0
N_SAMPLES     = int(SAMPLE_RATE * CLIP_DURATION)  # 16000

In [ ]:
# MFCC settings — same as the Arduino sketch
FRAME_LEN     = 256
HOP_LEN       = 128
N_MEL         = 26
N_MFCC        = 13
MEL_LOW_HZ    = 300.0
MEL_HIGH_HZ   = 8000.0
PRE_EMPHASIS  = 0.97

In [ ]:
# Classes
CLASSES   = ['clap', 'tap', 'snap', 'silence']
N_CLASSES = len(CLASSES)

In [ ]:
# Training config
TARGET_PER_CLASS = 250
EPOCHS           = 60
BATCH_SIZE       = 16
VAL_SPLIT        = 0.2
SEED             = 42

In [ ]:
# Number of MFCC frames per 1-second clip: (16000 - 256) / 128 + 1 = 124
NUM_FRAMES = (N_SAMPLES - FRAME_LEN) // HOP_LEN + 1
N_FFT_BINS = FRAME_LEN // 2 + 1  # 129

In [ ]:
print(f'Config OK — MFCC shape per clip: ({N_MFCC}, {NUM_FRAMES})')
assert NUM_FRAMES == 124, f'Expected 124 frames, got {NUM_FRAMES}'

In [ ]:
"""## 2 — Load dataset

Upload a ZIP file organized like this:
```
dataset.zip
├── clap/     (wav or mp4 files)
├── tap/
├── snap/
└── silence/
```
"""

In [ ]:
import zipfile
from google.colab import files

In [ ]:
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('dataset')

In [ ]:
# Find the folder that contains our class subfolders
DATA_ROOT = 'dataset'
for root, dirs, _ in os.walk('dataset'):
    if any(c in dirs for c in CLASSES):
        DATA_ROOT = root
        break

In [ ]:
print(f'Dataset root: {DATA_ROOT}')
for cls in CLASSES:
    p = os.path.join(DATA_ROOT, cls)
    os.makedirs(p, exist_ok=True)
    files_all = os.listdir(p) if os.path.exists(p) else []
    n_wav = len([f for f in files_all if f.lower().endswith('.wav')])
    n_mp4 = len([f for f in files_all if f.lower().endswith('.mp4')])
    print(f'  {cls}: {n_wav} wav, {n_mp4} mp4')

In [ ]:
"""## 3 — Convert MP4 to WAV

Some of our recordings were in mp4 (recorded on phones), so we convert them here.
"""

In [ ]:
from pydub import AudioSegment

In [ ]:
for cls in CLASSES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    mp4_files = [f for f in os.listdir(cls_dir) if f.lower().endswith('.mp4')]
    for mp4 in mp4_files:
        src = os.path.join(cls_dir, mp4)
        dst = os.path.join(cls_dir, os.path.splitext(mp4)[0] + '.wav')
        audio = AudioSegment.from_file(src, format='mp4')
        audio = audio.set_channels(1).set_frame_rate(SAMPLE_RATE)
        audio.export(dst, format='wav')
    if mp4_files:
        print(f'  {cls}: converted {len(mp4_files)} mp4 files')

In [ ]:
print('\nAvailable wav files:')
for cls in CLASSES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    n = len([f for f in os.listdir(cls_dir)
             if f.lower().endswith('.wav') and 'synth' not in f and 'aug' not in f])
    print(f'  {cls}: {n} original files')

In [ ]:
"""## 4 — Data Augmentation

We only have ~10 recordings per class, which is not enough to train a CNN.
So we generate 250 samples per class by applying random transformations:
white noise, time stretching, pitch shifting, time shifting, and volume changes.
"""

In [ ]:
import librosa
import soundfile as sf

In [ ]:
def load_and_fix(path):
    """Load a wav file, resample to 16kHz mono, pad/trim to exactly 1 second."""
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=CLIP_DURATION)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    return y.astype(np.float32)

In [ ]:
def augment_sample(y, rng, choice):
    """Apply a random augmentation to an audio sample."""
    y = y.copy()
    op = choice % 6
    if op == 0:
        y += rng.normal(0, rng.uniform(0.003, 0.012), len(y)).astype(np.float32)
    elif op == 1:
        rate = rng.uniform(0.82, 1.18)
        y = librosa.effects.time_stretch(y, rate=rate)
        y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    elif op == 2:
        steps = float(rng.choice([-2, -1, 1, 2]))
        y = librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=steps)
    elif op == 3:
        shift = int(rng.uniform(0.05, 0.25) * SAMPLE_RATE)
        y = np.roll(y, shift)
    elif op == 4:
        y = (y * rng.uniform(0.5, 1.5)).astype(np.float32)
    elif op == 5:
        y += rng.normal(0, 0.005, len(y)).astype(np.float32)
        y = (y * rng.uniform(0.7, 1.3)).astype(np.float32)
    return y.astype(np.float32)

In [ ]:
rng = np.random.default_rng(SEED)

In [ ]:
for cls in CLASSES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    originals = [f for f in os.listdir(cls_dir)
                 if f.lower().endswith('.wav') and 'aug' not in f]

    if not originals:
        print(f'  {cls}: no original files found, skipping')
        continue

    originals_audio = [load_and_fix(os.path.join(cls_dir, f)) for f in originals]

    # Clean up old augmented files if re-running
    for f in os.listdir(cls_dir):
        if 'aug' in f and f.endswith('.wav'):
            os.remove(os.path.join(cls_dir, f))

    generated = 0
    while generated < TARGET_PER_CLASS:
        src = originals_audio[generated % len(originals_audio)]
        y_aug = augment_sample(src, rng, generated)
        sf.write(os.path.join(cls_dir, f'{cls}_aug_{generated:04d}.wav'),
                 y_aug, SAMPLE_RATE)
        generated += 1

    total = len([f for f in os.listdir(cls_dir) if f.endswith('.wav')])
    print(f'  {cls}: {len(originals)} original + {generated} augmented = {total} total')

In [ ]:
print('\nAugmentation done.')

In [ ]:
"""## 5 — MFCC Extraction (CMSIS-compatible)

This is the most important part. We need the MFCC values computed here to match
what the Arduino produces with CMSIS-DSP. That means:

- No `librosa.feature.mfcc()` (it uses Slaney norm + ortho DCT internally)
- We use `scipy.fft.rfft` which behaves the same as `arm_rfft_fast_f32`
- Triangular mel filterbank without any normalization
- Plain DCT-II without the orthonormal scaling factor
- Pre-emphasis applied per-frame, not on the whole signal
- Natural log, not log10

We spent a while debugging this — the first version used librosa and the Arduino
output was always ~25% across all classes because the feature values were in a
completely different range.
"""

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy.fft import rfft
from scipy.signal import get_window

In [ ]:
# Hamming window — same formula as the Arduino sketch
_hamming = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(FRAME_LEN) / (FRAME_LEN - 1))
_hamming = _hamming.astype(np.float32)

In [ ]:
# Build the mel filterbank manually (no Slaney normalization)
def _build_mel_fb():
    def hz2mel(h): return 2595.0 * np.log10(1.0 + h / 700.0)
    def mel2hz(m): return 700.0 * (10.0 ** (m / 2595.0) - 1.0)

    mel_low  = hz2mel(MEL_LOW_HZ)
    mel_high = hz2mel(MEL_HIGH_HZ)
    mel_pts  = np.linspace(mel_low, mel_high, N_MEL + 2)

    # Bin indices — same as Arduino: floor((FRAME_LEN+1) * hz / SR)
    bin_idx  = np.floor((FRAME_LEN + 1) * mel2hz(mel_pts) / SAMPLE_RATE).astype(int)
    bin_idx  = np.clip(bin_idx, 0, N_FFT_BINS - 1)

    fb = np.zeros((N_MEL, N_FFT_BINS), dtype=np.float32)
    for m in range(N_MEL):
        lo, ctr, hi = bin_idx[m], bin_idx[m+1], bin_idx[m+2]
        if ctr != lo:
            for k in range(lo, ctr):
                fb[m, k] = (k - lo) / (ctr - lo)
        if hi != ctr:
            for k in range(ctr, hi + 1):
                fb[m, k] = (hi - k) / (hi - ctr)
    return fb

In [ ]:
_mel_fb = _build_mel_fb()  # shape (26, 129)

In [ ]:
# DCT-II matrix without orthonormal normalization
# dct[k] = sum_m cos(pi * k * (2m+1) / (2*N_MEL)) * log_mel[m]
_m_idx = np.arange(N_MEL, dtype=np.float32)
_k_idx = np.arange(N_MFCC, dtype=np.float32)
_dct_mat = np.cos(np.pi * _k_idx[:, None] * (2 * _m_idx[None, :] + 1)
                  / (2 * N_MEL)).astype(np.float32)  # (13, 26)

In [ ]:
def extract_mfcc(path):
    """
    Extract MFCC features using the same pipeline as the Arduino sketch.
    Returns a (13, 124) float32 array.
    """
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=CLIP_DURATION)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    y = y.astype(np.float32)

    mfcc_out = np.zeros((N_MFCC, NUM_FRAMES), dtype=np.float32)

    for f in range(NUM_FRAMES):
        frame = y[f * HOP_LEN : f * HOP_LEN + FRAME_LEN].copy()

        # Pre-emphasis (per-frame, like the Arduino does it)
        sig = np.empty(FRAME_LEN, dtype=np.float32)
        sig[0] = frame[0]
        sig[1:] = frame[1:] - PRE_EMPHASIS * frame[:-1]

        # Hamming window
        windowed = sig * _hamming

        # FFT — scipy rfft, no normalization (same as arm_rfft_fast_f32)
        X = rfft(windowed, n=FRAME_LEN)

        # Power spectrum
        power = (X.real ** 2 + X.imag ** 2).astype(np.float32)

        # Mel filterbank
        mel_e = _mel_fb @ power

        # Log (natural log, not log10)
        log_mel = np.log(mel_e + 1e-9)

        # DCT
        mfcc_out[:, f] = _dct_mat @ log_mel

    return mfcc_out

In [ ]:
# Quick check that the shape is correct
for cls in CLASSES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    wavs = [f for f in os.listdir(cls_dir) if f.endswith('.wav')]
    if wavs:
        sample_mfcc = extract_mfcc(os.path.join(cls_dir, wavs[0]))
        print(f'MFCC shape ({cls}): {sample_mfcc.shape}  (expected ({N_MFCC}, {NUM_FRAMES}))')
        assert sample_mfcc.shape == (N_MFCC, NUM_FRAMES)
        break

In [ ]:
# Plot one sample per class
fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 3))
for ax, cls in zip(axes, CLASSES):
    cls_dir = os.path.join(DATA_ROOT, cls)
    wavs = [f for f in os.listdir(cls_dir) if f.endswith('.wav')]
    if wavs:
        m = extract_mfcc(os.path.join(cls_dir, wavs[0]))
        librosa.display.specshow(m, ax=ax, sr=SAMPLE_RATE, hop_length=HOP_LEN)
        ax.set_title(cls)
plt.suptitle('MFCC features (CMSIS-DSP pipeline)')
plt.tight_layout(); plt.show()

In [ ]:
# Extract MFCCs for the whole dataset
from tqdm import tqdm

In [ ]:
X_list, y_list = [], []

In [ ]:
for label_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATA_ROOT, cls)
    wavs = [f for f in os.listdir(cls_dir) if f.lower().endswith('.wav')]
    print(f'Processing {cls} ({len(wavs)} files)...')
    ok = 0
    for wav in tqdm(wavs, leave=False):
        try:
            mfcc = extract_mfcc(os.path.join(cls_dir, wav))
            X_list.append(mfcc)
            y_list.append(label_idx)
            ok += 1
        except Exception as e:
            print(f'  Skipped {wav}: {e}')
    print(f'  {ok} extracted')

In [ ]:
X = np.array(X_list)
y = np.array(y_list)
print(f'\nFinal dataset: X={X.shape}, y={y.shape}')
for i, cls in enumerate(CLASSES):
    print(f'  [{i}] {cls}: {np.sum(y==i)} samples')

In [ ]:
"""## 6 — Normalization and train/val split

Each MFCC coefficient lives in a different range (coeff 0 is around -30 to -50,
while higher coefficients are close to 0). So we normalize per-coefficient
using mean and std computed across all samples and all frames.

These mean/std values get baked into model.h so the Arduino can apply the
same normalization at inference time.
"""

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [ ]:
# Per-coefficient normalization
# X shape: (N, 13, 124) -> mean/std shape: (1, 13, 1)
X_mean = X.mean(axis=(0, 2), keepdims=True)
X_std  = X.std(axis=(0, 2),  keepdims=True) + 1e-8

In [ ]:
X_norm = (X - X_mean) / X_std
print(f'Per-coefficient normalization:')
print(f'  mean range: [{X_mean.min():.2f}, {X_mean.max():.2f}]')
print(f'  std  range: [{X_std.min():.4f}, {X_std.max():.4f}]')

In [ ]:
X_cnn = X_norm[..., np.newaxis]  # add channel dim -> (N, 13, 124, 1)
y_oh  = tf.keras.utils.to_categorical(y, num_classes=N_CLASSES)

In [ ]:
min_count = min(np.sum(y == i) for i in range(N_CLASSES))
use_stratify = min_count >= 2

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_cnn, y_oh,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=(y if use_stratify else None)
)

In [ ]:
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
"""## 7 — CNN model

We kept the architecture small on purpose — the Nano 33 BLE only has 256 KB of RAM
and we need the model to fit in ~34 KB of tensor arena. Two conv blocks with
batch norm, then global average pooling and a small dense layer.

We tried a third conv block initially but it was overfitting badly with our small dataset.
"""

In [ ]:
from tensorflow.keras import layers, models, callbacks, regularizers

In [ ]:
def build_model(input_shape, n_classes):
    inp = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(16, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    # Block 2
    x = layers.Conv2D(32, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    return models.Model(inp, out, name='mfcc_cnn_light')

In [ ]:
model = build_model(X_cnn.shape[1:], N_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
"""## 8 — Training

We also do some light augmentation directly on the MFCC tensors during training:
gaussian noise and a simple version of SpecAugment (masking random frequency bands).
This helps a bit with generalization since our dataset is still pretty small.
"""

In [ ]:
def mfcc_augment(x, y):
    # Add gaussian noise to the normalized MFCCs
    noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.05)
    x = x + noise
    # Mask 1-2 random MFCC rows (like SpecAugment but simpler)
    mask_size = tf.random.uniform([], 1, 3, dtype=tf.int32)
    mask_start = tf.random.uniform([], 0, N_MFCC - mask_size, dtype=tf.int32)
    mask = tf.ones([mask_start, NUM_FRAMES, 1])
    mask = tf.concat([mask,
                      tf.zeros([mask_size, NUM_FRAMES, 1]),
                      tf.ones([N_MFCC - mask_start - mask_size, NUM_FRAMES, 1])], axis=0)
    x = x * mask
    return x, y

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
train_ds = (tf.data.Dataset
            .from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=SEED)
            .map(mfcc_augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

In [ ]:
val_ds = (tf.data.Dataset
          .from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=5, min_lr=1e-5, verbose=1)
]

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=cb_list,
    verbose=1
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['accuracy'],     label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].set_ylim(0, 1); axes[0].legend()
axes[1].plot(history.history['loss'],     label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'\nVal accuracy: {val_acc*100:.1f}%  |  Val loss: {val_loss:.4f}')

In [ ]:
"""## 9 — Confusion Matrix"""

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
y_true     = np.argmax(y_val, axis=1)
y_pred_cls = np.argmax(model.predict(X_val, verbose=0), axis=1)

In [ ]:
print(classification_report(y_true, y_pred_cls, target_names=CLASSES))

In [ ]:
cm = confusion_matrix(y_true, y_pred_cls)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
"""## 10 — Export to TFLite INT8

We quantize everything to INT8 (weights + activations) so it runs efficiently
on the Cortex-M4. The representative dataset is needed for calibration —
TFLite uses it to figure out the right scale/zero_point for each layer.
"""

In [ ]:
def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

In [ ]:
tflite_model = converter.convert()

In [ ]:
TFLITE_PATH = 'keyword_model.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
print(f'Saved {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH)/1024:.1f} KB)')

In [ ]:
# Print quantization parameters (we'll need these for the Arduino sketch)
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
in_det  = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]
in_scale,  in_zp  = in_det['quantization']
out_scale, out_zp = out_det['quantization']
print(f'Input  scale={in_scale:.6f}, zero_point={in_zp}')
print(f'Output scale={out_scale:.6f}, zero_point={out_zp}')

In [ ]:
"""### 10b — Verify INT8 accuracy

Quick sanity check: run the quantized model on the validation set and make sure
accuracy didn't drop too much compared to the float model.
"""

In [ ]:
correct = 0
for i in range(len(X_val)):
    inp_q = np.round(X_val[i:i+1] / in_scale + in_zp).clip(-128, 127).astype(np.int8)
    interpreter.set_tensor(in_det['index'], inp_q)
    interpreter.invoke()
    out_q = interpreter.get_tensor(out_det['index'])
    pred  = np.argmax((out_q.astype(np.float32) - out_zp) * out_scale)
    if pred == y_true[i]: correct += 1

In [ ]:
print(f'TFLite INT8 accuracy: {correct}/{len(X_val)} = {100*correct/len(X_val):.1f}%')

In [ ]:
"""## 11 — Generate model.h

This creates the C header file that goes into the Arduino sketch folder.
It contains the TFLite model as a byte array (via xxd), plus all the
parameters the sketch needs: MFCC settings, normalization constants,
quantization params, and class labels.
"""

In [ ]:
# Convert tflite to C array using xxd
!echo "const unsigned char model[] = {" > model.h
!cat keyword_model.tflite | xxd -i          >> model.h
!echo "};"                                  >> model.h

In [ ]:
# Now prepend the header with all the #defines and constants
with open('model.h', 'r') as f:
    xxd_body = f.read()

In [ ]:
mean_flat = X_mean.flatten().astype('float32')
std_flat  = X_std.flatten().astype('float32')
mean_str  = ', '.join(f'{v:.8f}f' for v in mean_flat)
std_str   = ', '.join(f'{v:.8f}f' for v in std_flat)
classes_str = ', '.join(f'"{c}"' for c in CLASSES)

In [ ]:
header = f"""// model.h — generated by the training notebook
// Copy this file into the Arduino sketch folder.
#pragma once
#include <stdint.h>

// MFCC parameters (must match the Arduino sketch)
#define SAMPLE_RATE        {SAMPLE_RATE}
#define FRAME_LEN          {FRAME_LEN}
#define HOP_LEN            {HOP_LEN}
#define N_MEL              {N_MEL}
#define N_MFCC             {N_MFCC}
#define MEL_LOW_HZ         {MEL_LOW_HZ:.1f}f
#define MEL_HIGH_HZ        {MEL_HIGH_HZ:.1f}f
#define PRE_EMPHASIS       {PRE_EMPHASIS:.2f}f
#define CLIP_SAMPLES       {N_SAMPLES}

// Per-coefficient normalization (computed from training data)
// On Arduino: norm = (mfcc[c] - NORM_MEAN[c]) / NORM_STD[c]
static const float NORM_MEAN[N_MFCC] = {{ {mean_str} }};
static const float NORM_STD [N_MFCC] = {{ {std_str}  }};

// TFLite INT8 quantization parameters
#define INPUT_SCALE        {in_scale:.8f}f
#define INPUT_ZERO_POINT   {in_zp}
#define OUTPUT_SCALE       {out_scale:.8f}f
#define OUTPUT_ZERO_POINT  {out_zp}

// Class labels
#define N_CLASSES          {N_CLASSES}
static const char* const CLASS_LABELS[] = {{ {classes_str} }};

// TFLite model weights (xxd dump)
"""

In [ ]:
with open('model.h', 'w') as f:
    f.write(header + xxd_body)

In [ ]:
print(f'model.h ready: {os.path.getsize("model.h")/1024:.1f} KB')
!head -35 model.h

In [ ]:
"""## 12 — Download files"""

In [ ]:
from google.colab import files as colab_files
colab_files.download('keyword_model.tflite')
colab_files.download('model.h')
print('Downloads started — copy model.h into the Arduino sketch folder.')

In [ ]:
"""## 13 — Quick test on a single audio file

Upload a wav or mp4 and see what the model predicts. Useful for sanity checking
before flashing the Arduino.
"""

In [ ]:
from google.colab import files
uploaded_test = files.upload()
test_path = list(uploaded_test.keys())[0]

In [ ]:
# Convert mp4 if needed
if test_path.lower().endswith('.mp4'):
    audio = AudioSegment.from_file(test_path, format='mp4')
    audio = audio.set_channels(1).set_frame_rate(SAMPLE_RATE)
    test_path_wav = test_path.replace('.mp4', '_test.wav')
    audio.export(test_path_wav, format='wav')
    test_path = test_path_wav

In [ ]:
# Run through the same pipeline
mfcc_test = extract_mfcc(test_path)
mfcc_norm = (mfcc_test - X_mean[0, :, 0:1]) / X_std[0, :, 0:1]
mfcc_inp  = mfcc_norm[np.newaxis, ..., np.newaxis].astype(np.float32)
mfcc_q    = np.round(mfcc_inp / in_scale + in_zp).clip(-128, 127).astype(np.int8)

In [ ]:
interpreter.set_tensor(in_det['index'], mfcc_q)
interpreter.invoke()
out_q = interpreter.get_tensor(out_det['index'])[0]
probs = (out_q.astype(np.float32) - out_zp) * out_scale

In [ ]:
pred_class = CLASSES[np.argmax(probs)]
print(f'\nPredicted: {pred_class.upper()}')
for cls, p in zip(CLASSES, probs):
    bar = '█' * int(max(p, 0) * 30)
    print(f'  {cls:8s}: {p:.4f}  {bar}')